In [9]:
import numpy as np
import skfuzzy as fuzzy
from skfuzzy import control as ctrl

In [10]:
FC_MIN, FC_MAX = 40, 160
SPO2_MIN, SPO2_MAX = 70, 100


def criar_sistema_triagem_uti():
    """Monta o sistema fuzzy Mamdani de triagem de risco clínico da UTI."""
    # 1. Antecedentes e consequente
    freq_cardiaca = ctrl.Antecedent(np.arange(FC_MIN, FC_MAX + 1, 1), 'freq_cardiaca')
    spo2 = ctrl.Antecedent(np.arange(SPO2_MIN, SPO2_MAX + 1, 1), 'spo2')
    risco = ctrl.Consequent(np.arange(0, 101, 1), 'risco')

    # 2. Defuzzificação
    risco.defuzzify_method = 'centroid'

    # 3. Funções de pertinência
    freq_cardiaca['bradicardia'] = fuzzy.trimf(freq_cardiaca.universe, [40, 40, 60])
    freq_cardiaca['normal'] = fuzzy.trimf(freq_cardiaca.universe, [50, 75, 100])
    freq_cardiaca['taquicardia'] = fuzzy.trapmf(freq_cardiaca.universe, [90, 110, 160, 160])

    spo2['critica'] = fuzzy.trapmf(spo2.universe, [70, 70, 88, 91])
    spo2['limitrofe'] = fuzzy.trimf(spo2.universe, [89, 93, 96])
    spo2['normal'] = fuzzy.trapmf(spo2.universe, [94, 97, 100, 100])

    risco['baixo'] = fuzzy.trimf(risco.universe, [0, 0, 35])
    risco['moderado'] = fuzzy.trimf(risco.universe, [25, 50, 75])
    risco['critico'] = fuzzy.trapmf(risco.universe, [65, 85, 100, 100])

    # 4. Regras clínicas
    regras = [
        ctrl.Rule(spo2['critica'], risco['critico']),                                    # R1
        ctrl.Rule(freq_cardiaca['bradicardia'] & spo2['limitrofe'], risco['critico']),   # R2
        ctrl.Rule(freq_cardiaca['taquicardia'] & spo2['limitrofe'], risco['critico']),   # R3
        ctrl.Rule(freq_cardiaca['normal'] & spo2['limitrofe'], risco['moderado']),       # R4
        ctrl.Rule(freq_cardiaca['normal'] & spo2['normal'], risco['baixo']),             # R5
    ]

    # 5. Simulador compilado
    return ctrl.ControlSystemSimulation(ctrl.ControlSystem(regras))


def avaliar_paciente(simulador, bpm, spo2_val):
    """Retorna o risco clínico (0-100%) para uma leitura de FC (bpm) e SpO2 (%)."""
    if not (FC_MIN <= bpm <= FC_MAX):
        raise ValueError(f"bpm fora do intervalo fisiológico [{FC_MIN}, {FC_MAX}]: {bpm}")
    if not (SPO2_MIN <= spo2_val <= SPO2_MAX):
        raise ValueError(f"SpO2 fora do intervalo fisiológico [{SPO2_MIN}, {SPO2_MAX}]: {spo2_val}")

    simulador.input['freq_cardiaca'] = bpm
    simulador.input['spo2'] = spo2_val
    try:
        simulador.compute()
        return simulador.output['risco']
    except (KeyError, ValueError) as exc:
        # Nenhuma das 5 regras foi ativada (ex.: FC 130 com SpO2 98%): combinação não coberta.
        raise RuntimeError(
            f"Nenhuma regra clínica ativada para bpm={bpm}, SpO2={spo2_val}: "
            "combinação não coberta pela base de 5 regras."
        ) from exc


if __name__ == "__main__":
    sim = criar_sistema_triagem_uti()
    print("Sistema de UTI instanciado com sucesso!")

Sistema de UTI instanciado com sucesso!
